# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SafeeAkmal/FlyRank-ML-internship-tasks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content_hash_id, aggregated over the trailing 90-day window ending at the snapshot's last available date (2026-04-02 to 2026-06-30).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata
hf_token = userdata.get('hf_token')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}')")
WINDOW_START = "2026-04-02"
WINDOW_END   = "2026-06-30"   # freshness-lag cutoff per the guide

# Point this at wherever the parquet files actually live in the HF repo —
# check the dataset's file listing / notebook 03 for the exact path pattern.
fact_path = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance*.parquet"

query = f"""
SELECT
    content_hash_id,
    client_hash_id,
    COUNT(*)                                                      AS n_days_in_window,
    SUM(gsc_impressions) FILTER (WHERE gsc_data_available)        AS impressions_90d,
    SUM(gsc_clicks)      FILTER (WHERE gsc_data_available)        AS clicks_90d,
    SUM(gsc_clicks)  * 1.0 / NULLIF(SUM(gsc_impressions), 0)      AS ctr_90d,
    SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position_90d,   -- impression-weighted
    SUM(ga4_sessions)          FILTER (WHERE ga4_data_available)  AS sessions_90d,
    SUM(ga4_engaged_sessions)  FILTER (WHERE ga4_data_available)  AS engaged_sessions_90d,
    SUM(ga4_engaged_sessions) * 1.0 / NULLIF(SUM(ga4_sessions),0) AS engagement_rate_90d,
    SUM(ga4_total_engagement_sec) * 1.0 / NULLIF(SUM(ga4_sessions),0) AS avg_engagement_sec_90d,
    SUM(scroll_events)         FILTER (WHERE ga4_data_available)  AS scroll_events_90d
FROM read_parquet('{fact_path}')
WHERE report_date BETWEEN '{WINDOW_START}' AND '{WINDOW_END}'
GROUP BY content_hash_id, client_hash_id
"""
df = con.execute(query).df()

# --- the actual verification ---
n_rows = len(df)
n_unique_keys = df[["content_hash_id", "client_hash_id"]].drop_duplicates().shape[0]

print(f"Rows in aggregated table: {n_rows}")
print(f"Unique (content_hash_id, client_hash_id) pairs: {n_unique_keys}")
assert n_rows == n_unique_keys, "Grain check failed — duplicate content/client rows found!"
print(" Grain confirmed: one row per content item per client, over the 90-day window.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in aggregated table: 409205
Unique (content_hash_id, client_hash_id) pairs: 409205
 Grain confirmed: one row per content item per client, over the 90-day window.


In [7]:
print(df["n_days_in_window"].describe())         # should max out around 90
print((df["impressions_90d"] == 0).mean())        # fraction with zero real GSC volume

count    409205.000000
mean         28.577539
std           3.822835
min           3.000000
25%          30.000000
50%          30.000000
75%          30.000000
max          36.000000
Name: n_days_in_window, dtype: float64
0.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features come from fact_content_daily_performance (impressions, clicks, position, sessions, engagement, scroll) and dim_content (content type, search volume, competition, intent, word count, age). The label is a computed CTR-vs-position-tier gap, not a stored column. Context fields (keys, availability flags, publish status) filter and explain the ranking but never enter the score. Excluded: any product-decision-shaped field (optimization_eligible_date), content-generation metadata (provider_used, model_used), and deleted/unpublished rows.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata
hf_token = userdata.get('hf_token')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}')")

MONTH = "2026-03"
fact_path    = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/data_0.parquet"
content_path = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

query = f"""
WITH agg_fact AS (
    SELECT
        content_hash_id,
        client_hash_id,
        COUNT(*) FILTER (WHERE gsc_data_available)                    AS gsc_days,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available)        AS impressions_30d,
        SUM(gsc_clicks)      FILTER (WHERE gsc_data_available)        AS clicks_30d,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)       AS ctr_30d,
        SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions),0)  AS avg_position_30d,
        SUM(ga4_sessions)         FILTER (WHERE ga4_data_available)   AS sessions_30d,
        SUM(ga4_engaged_sessions) FILTER (WHERE ga4_data_available)   AS engaged_sessions_30d
    FROM read_parquet('{fact_path}')
    GROUP BY content_hash_id, client_hash_id
)
SELECT
    f.*,
    c.content_type, c.search_volume, c.competition, c.main_intent,
    c.word_count, c.content_created_date, c.content_updated_date,
    c.is_published, c.is_deleted
FROM agg_fact f
JOIN read_parquet('{content_path}') c USING (content_hash_id)
WHERE c.is_published = TRUE AND c.is_deleted = FALSE
"""

feature_frame = con.execute(query).df()
print(feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(321106, 18)


,content_hash_id,client_hash_id,gsc_days,impressions_30d,clicks_30d,ctr_30d,avg_position_30d,sessions_30d,engaged_sessions_30d,content_type,search_volume,competition,main_intent,word_count,content_created_date,content_updated_date,is_published,is_deleted
0,content_67741cce996cfafa,client_62f4a7e64f5e0096,16,46.0,1.0,0.021739,4.543478,NaN,NaN,keyword article,10,0.00,informational,3057,2026-02-12,2026-06-29,True,False
1,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,31,899.0,1.0,0.001112,5.825362,NaN,NaN,keyword article,10,0.00,informational,2855,2026-02-12,2026-06-29,True,False
2,content_65c50dfe9d87a585,client_62f4a7e64f5e0096,30,3108.0,0.0,0.000000,6.953668,NaN,NaN,keyword article,10,0.00,informational,2779,2026-02-12,2026-06-29,True,False
3,content_275b6f7f733016d4,client_62f4a7e64f5e0096,29,810.0,1.0,0.001235,4.624691,NaN,NaN,keyword article,10,0.00,informational,3653,2026-02-12,2026-06-29,True,False
4,content_4dc944b7d0b65ecc,client_62f4a7e64f5e0096,26,134.0,0.0,0.000000,4.865672,NaN,NaN,keyword article,10,0.62,informational,3698,2026-02-12,2026-06-29,True,False


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Query A: GRAIN — one row really is one content_hash_id x client_hash_id per day ---
grain_check = con.execute(f"""
    SELECT content_hash_id, client_hash_id, COUNT(*) AS n
    FROM read_parquet('{fact_path}')
    GROUP BY content_hash_id, client_hash_id
    HAVING COUNT(*) > 31   -- more rows than days in March = grain violation
""").df()
print("Rows violating expected daily grain:", len(grain_check))

# --- Query B: COUNT + DATE SPAN of the slice ---
span_check = con.execute(f"""
    SELECT
        COUNT(*)                         AS total_rows,
        COUNT(DISTINCT content_hash_id)  AS unique_content,
        MIN(report_date)                 AS min_date,
        MAX(report_date)                 AS max_date
    FROM read_parquet('{fact_path}')
""").df()
span_check

# --- Query C: AVAILABILITY — filter with IS TRUE, show survival count ---
availability_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{fact_path}')
""").df()
availability_check

# --- Query D: MISSING VALUES in the feature frame from Snippet 1 ---
print(feature_frame[[
    "impressions_30d","ctr_30d","avg_position_30d",
    "sessions_30d","word_count","content_type"
]].isna().mean().sort_values(ascending=False))

Rows violating expected daily grain: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

sessions_30d        0.724178
impressions_30d     0.450126
ctr_30d             0.450126
avg_position_30d    0.450126
word_count          0.329053
content_type        0.000000
dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Causation. No matter how strong a CTR gap or engagement pattern looks, this data can only show association. We cannot say a low CTR caused a page to lose traffic, or that fixing metadata will recover clicks that requires an actual experiment (A/B test, before/after intervention with a control group), which this snapshot doesn't contain.

Anything about clients whose tracking hadn't started yet. Before ga4_data_start, a client's rows have ga4_data_available = FALSE meaning GA4 metrics aren't "zero engagement," they're simply unmeasured. Treating that as real zero-engagement would fabricate a signal that doesn't exist.

Fair comparison across clients with different history lengths. This is an unbalanced panel only 9 of 70 clients have 12+ months of data. Any month-over-month or seasonal claim you make will be dominated by whichever clients happen to have long histories, not by all clients equally.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata
hf_token = userdata.get('hf_token')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}')")
MONTH = "2026-03"
fact_path    = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/data_0.parquet"
MONTH = "2026-03"
fact_path = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/data_0.parquet"

partial_month_check = con.execute(f"""
    WITH daily_flags AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_data_available,
            ga4_data_available
        FROM read_parquet('{fact_path}')
    ),
    per_page AS (
        SELECT
            content_hash_id,
            client_hash_id,
            COUNT(*) AS days_in_march,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_days,
            COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_days
        FROM daily_flags
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        COUNT(*) AS total_pages,
        COUNT(*) FILTER (WHERE gsc_available_days > 0 AND gsc_available_days < days_in_march) AS pages_with_partial_gsc,
        COUNT(*) FILTER (WHERE ga4_available_days > 0 AND ga4_available_days < days_in_march) AS pages_with_partial_ga4,
        ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_available_days > 0 AND gsc_available_days < days_in_march) / COUNT(*), 1) AS pct_partial_gsc,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_available_days > 0 AND ga4_available_days < days_in_march) / COUNT(*), 1) AS pct_partial_ga4
    FROM per_page
""").df()

print(partial_month_check.to_string(index=False))

pct_ga4 = partial_month_check["pct_partial_ga4"].iloc[0]
pct_gsc = partial_month_check["pct_partial_gsc"].iloc[0]

print(f"""
LIMITATION:
{pct_ga4}% of pages in the {MONTH} slice have partial-month GA4 tracking,
and {pct_gsc}% have partial-month GSC tracking meaning their tracking
started mid-month rather than on day 1. Raw sums like sessions_30d or
impressions_30d understate true volume for these pages specifically,
which would bias any ranking built on raw totals against newly-tracked
content. This is a structural feature of the unbalanced panel (per-client
gsc_data_start/ga4_data_start), not noise it cannot be fixed by more
data, only accounted for (e.g. by adjusting for available-days count or
excluding partial-month pages from comparison).
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_pages  pages_with_partial_gsc  pages_with_partial_ga4  pct_partial_gsc  pct_partial_ga4
      331437                  114873                   90306             34.7             27.2

LIMITATION:
27.2% of pages in the 2026-03 slice have partial-month GA4 tracking,
and 34.7% have partial-month GSC tracking meaning their tracking
started mid-month rather than on day 1. Raw sums like sessions_30d or
impressions_30d understate true volume for these pages specifically,
which would bias any ranking built on raw totals against newly-tracked
content. This is a structural feature of the unbalanced panel (per-client
gsc_data_start/ga4_data_start), not noise it cannot be fixed by more
data, only accounted for (e.g. by adjusting for available-days count or
excluding partial-month pages from comparison).



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.